In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import subprocess, os, shutil

USB_PATH = '/kaggle/working/USB'
CG_PATH  = '/kaggle/working/CGMatch'

os.system(f"rm -rf {USB_PATH} {CG_PATH}")
os.system(f"git clone https://github.com/microsoft/Semi-supervised-learning {USB_PATH}")
os.system(f"git clone https://github.com/Alam8010/CGMatch.git {CG_PATH}")

shutil.copytree(f'{CG_PATH}/semilearn', f'{USB_PATH}/semilearn', dirs_exist_ok=True)
shutil.copytree(f'{CG_PATH}/config',    f'{USB_PATH}/config',    dirs_exist_ok=True)
shutil.copy(f'{CG_PATH}/train.py', f'{USB_PATH}/train.py')

os.system("pip install timm==0.6.12 progress ruamel.yaml tensorboard tqdm scikit-learn -q")

r = subprocess.run(
    ['python', '-c', 'from semilearn.algorithms import get_algorithm; print("Import OK ✅")'],
    cwd=USB_PATH, capture_output=True, text=True
)
print(r.stdout.strip() or r.stderr.strip())

Cloning into '/kaggle/working/USB'...
Cloning into '/kaggle/working/CGMatch'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 13.3 MB/s eta 0:00:00
Import OK ✅


In [4]:
with open('/kaggle/working/USB/config/classic_cv/fixmatch/fixmatch_cifar10_40_0.yaml', 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if   'num_train_iter' in line: new_lines.append('num_train_iter: 16384\n')
    elif 'warm_up_iter'   in line: new_lines.append('warm_up_iter: 32\n')
    elif 'num_eval_iter'  in line: new_lines.append('num_eval_iter: 512\n')
    elif line.strip().startswith('gpu:'): new_lines.append('gpu: 0\n')
    else: new_lines.append(line)

with open('/kaggle/working/USB/config/classic_cv/fixmatch/fixmatch_cifar10_40_0.yaml', 'w') as f:
    f.writelines(new_lines)

print("YAML patched ✅")

YAML patched ✅


In [ ]:
!cd /kaggle/working/USB && CUDA_VISIBLE_DEVICES=0 python train.py \
    --c config/classic_cv/fixmatch/fixmatch_cifar10_40_0.yaml \
    --save_dir /kaggle/working/results

/kaggle/working/USB/train.py:286: UserWarning: You have chosen to seed training. This will turn on the CUDNN deterministic setting, which can slow down your training considerably! You may see unexpected behavior when restarting from checkpoints.
  warnings.warn(
/kaggle/working/USB/train.py:297: UserWarning: You have chosen a specific GPU. This will completely disable data parallelism.
  warnings.warn(
[2026-10-20 05:10:29,854 INFO] Use GPU: 0 for training
/kaggle/working/USB/semilearn/core/algorithmbase.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.loss_scaler = GradScaler()
100%|█████████████████████████████████████████| 170M/170M [25:09<00:00, 113kB/s]
lb count: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
ulb count: [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]
[2026-10-20 05:35:43,691 INFO] unlabeled data number: 50000, labeled data number 40
/usr/local/lib/python3.13/dist-packages/torc